## Fuente: MediaPipe Hand Landmarker

https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker/python

Requiere MediaPipe 1.0.1, OpenCV, una webcam y conexión a Internet para descargar el modelo al ejecutar la celda. Presionar Esc para salir.

![Hand landmarks](./img/hand_landmarks.png)

In [2]:
import time
from urllib.request import urlopen

import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision

mp_drawing = vision.drawing_utils
mp_drawing_styles = vision.drawing_styles
mp_hands = vision.HandLandmarksConnections

# Función para contar los dedos levantados
def count_fingers(hand_landmarks):
    finger_tips = [8, 12, 16, 20]  # Índices de los puntos clave de las puntas de los dedos
    finger_foldings = [6, 10, 14, 18]  # Índices de las articulaciones cercanas a la palma

    fingers_up = 0
    for tip, folding in zip(finger_tips, finger_foldings):
        if hand_landmarks[tip].y < hand_landmarks[folding].y:
            fingers_up += 1
    if hand_landmarks[4].x < hand_landmarks[3].x:
        fingers_up += 1

    return fingers_up

model_url = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/hand_landmarker.task"
)
print("Descargando el modelo de Hand Landmarker...")
with urlopen(model_url, timeout=60) as response:
    model_data = response.read()

options = vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_buffer=model_data),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

with vision.HandLandmarker.create_from_options(options) as hands:
    cap = cv2.VideoCapture(0)
    try:
        if not cap.isOpened():
            raise RuntimeError("No se pudo abrir la webcam.")

        previous_timestamp = -1
        while True:
            success, image = cap.read()
            if not success:
                raise RuntimeError("No se pudo leer un cuadro de la webcam.")

            rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
            timestamp = max(time.monotonic_ns() // 1_000_000, previous_timestamp + 1)
            previous_timestamp = timestamp
            results = hands.detect_for_video(mp_image, timestamp)

            fingers_count = 0
            for hand_landmarks in results.hand_landmarks:
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style())

                fingers_count = count_fingers(hand_landmarks)

            # Espejar la imagen
            flipped_image = cv2.flip(image, 1)

            # Agregar el texto
            cv2.putText(
                flipped_image, f"Dedos levantados: {fingers_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

            # Mostrar la imagen
            cv2.imshow('MediaPipe Hands', flipped_image)
            if cv2.waitKey(5) & 0xFF == 27:
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()


Descargando el modelo de Hand Landmarker...
